In [1]:
from unsloth import FastLanguageModel

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
import os
import json
import random
import shutil
from pathlib import Path
import warnings 
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoTokenizer
from peft import LoraConfig, get_peft_model
from transformers import DataCollatorForSeq2Seq
from trl import SFTTrainer, SFTConfig
from peft import PeftModel
import json
from tqdm.auto import tqdm
SEED = 322
warnings.filterwarnings('ignore')
def set_seed(seed: int = 322):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA capability:", torch.cuda.get_device_capability(0))

Torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA capability: (7, 5)


In [ ]:
# !pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:


CANDIDATE_DATA_DIRS = [
    Path("./processed_spider_sft"),
    Path("/kaggle/working/processed_spider_sft"),
]

# Дополнительно ищем в /kaggle/input
kaggle_input = Path("/kaggle/input/datasets/olllllllll/spider")
if kaggle_input.exists():
    CANDIDATE_DATA_DIRS.extend([p.parent for p in kaggle_input.glob("**/train_sft.jsonl")])

DATA_DIR = None
for candidate in CANDIDATE_DATA_DIRS:
    if (candidate / "train_sft.jsonl").exists() and (candidate / "dev_sft.jsonl").exists():
        DATA_DIR = candidate
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        "Не найдена папка processed_spider_sft с файлами train_sft.jsonl и dev_sft.jsonl. "
        "Положи processed_spider_sft рядом с ноутбуком или добавь её как Kaggle Dataset."
    )

TRAIN_PATH = DATA_DIR / "train_sft.jsonl"
DEV_PATH = DATA_DIR / "dev_sft.jsonl"
DEV_EVAL_PATH = DATA_DIR / "dev_eval_prompts.jsonl"

print("DATA_DIR:", DATA_DIR)
print("TRAIN_PATH:", TRAIN_PATH)
print("DEV_PATH:", DEV_PATH)
print("DEV_EVAL_PATH exists:", DEV_EVAL_PATH.exists())

DATA_DIR: /kaggle/input/datasets/olllllllll/spider
TRAIN_PATH: /kaggle/input/datasets/olllllllll/spider/train_sft.jsonl
DEV_PATH: /kaggle/input/datasets/olllllllll/spider/dev_sft.jsonl
DEV_EVAL_PATH exists: True


In [ ]:

dataset = load_dataset(
    "json",
    data_files={
        "train": str(TRAIN_PATH),
        "validation": str(DEV_PATH),
    },
)
print(dataset)
print("Train columns:", dataset["train"].column_names)
print("Validation columns:", dataset["validation"].column_names)
print("Train size:", len(dataset["train"]))
print("Validation size:", len(dataset["validation"]))

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'split', 'db_id', 'question', 'schema', 'sql', 'text'],
        num_rows: 8659
    })
    validation: Dataset({
        features: ['id', 'split', 'db_id', 'question', 'schema', 'sql', 'text'],
        num_rows: 1034
    })
})
Train columns: ['id', 'split', 'db_id', 'question', 'schema', 'sql', 'text']
Validation columns: ['id', 'split', 'db_id', 'question', 'schema', 'sql', 'text']
Train size: 8659
Validation size: 1034


In [ ]:
sample = dataset["train"][0]
print("Keys:", sample.keys())
print("=" * 80)
print(sample["text"][:3000])

Keys: dict_keys(['id', 'split', 'db_id', 'question', 'schema', 'sql', 'text'])
### Instruction:
You are a Text-to-SQL assistant. Generate a valid SQLite SQL query for the given question using only the provided database schema. Do not use tables or columns that are not present in the schema. Return only the SQL query.

### Database schema:
CREATE TABLE department (
    Department_ID NUMBER,
    Name TEXT,
    Creation TEXT,
    Ranking NUMBER,
    Budget_in_Billions NUMBER,
    Num_Employees NUMBER,
    PRIMARY KEY (Department_ID)
);

CREATE TABLE head (
    head_ID NUMBER,
    name TEXT,
    born_state TEXT,
    age NUMBER,
    PRIMARY KEY (head_ID)
);

CREATE TABLE management (
    department_ID NUMBER,
    head_ID NUMBER,
    temporary_acting TEXT,
    PRIMARY KEY (department_ID),
    FOREIGN KEY (head_ID) REFERENCES head(head_ID),
    FOREIGN KEY (department_ID) REFERENCES department(Department_ID)
);

### Question:
How many heads of the departments are older than 56 ?

### SQL:
SEL

In [ ]:
OUTPUT_DIR = Path("./qwen25-coder-15b-spider-qlora-r16")
ADAPTER_DIR = OUTPUT_DIR / "adapter"
LOG_DIR = OUTPUT_DIR / "logs"
HISTORY_PATH = OUTPUT_DIR / "training_history.csv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-Coder-1.5B-Instruct", 
    max_seq_length=2048,
    dtype=torch.float16,          
    load_in_4bit=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print("pad_token:", tokenizer.pad_token)
print("eos_token:", tokenizer.eos_token)
print("model_max_length:", tokenizer.model_max_length)

==((====))==  Unsloth 2026.6.1: Fast Qwen2 patching. Transformers: 5.10.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

pad_token: <|PAD_TOKEN|>
eos_token: <|im_end|>
model_max_length: 32768


In [ ]:

model = FastLanguageModel.get_peft_model(
    model,
    r=16,                      
    lora_alpha=32,   
    lora_dropout=0.0,         
    bias="none",                
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=322,             
)


Unsloth 2026.6.1 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [ ]:

MAX_SEQ_LENGTH = 2048
SQL_MARKER = "### SQL:\n"

def split_prompt_and_answer(text):
    if SQL_MARKER not in text:
        raise ValueError("SQL marker not found in text")

    prompt_part, answer_part = text.split(SQL_MARKER, 1)

    prompt = prompt_part + SQL_MARKER
    answer = answer_part.strip()

    return prompt, answer

def tokenize_with_sql_labels(example):
    prompt, answer = split_prompt_and_answer(example["text"])
    answer = answer + tokenizer.eos_token

    prompt_ids = tokenizer(
        prompt,
        add_special_tokens=False,
    )["input_ids"]

    answer_ids = tokenizer(
        answer,
        add_special_tokens=False,
    )["input_ids"]

    input_ids = prompt_ids + answer_ids
    attention_mask = [1] * len(input_ids)
    labels = [-100] * len(prompt_ids) + answer_ids.copy()
    if len(input_ids) > MAX_SEQ_LENGTH:
        input_ids = input_ids[:MAX_SEQ_LENGTH]
        attention_mask = attention_mask[:MAX_SEQ_LENGTH]
        labels = labels[:MAX_SEQ_LENGTH]
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "length": len(input_ids),
        "answer_tokens": sum(x != -100 for x in labels),
    }
tokenized_dataset = dataset.map(
    tokenize_with_sql_labels,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing with SQL-only labels",
)
print(tokenized_dataset)

Tokenizing with SQL-only labels:   0%|          | 0/8659 [00:00<?, ? examples/s]

Tokenizing with SQL-only labels:   0%|          | 0/1034 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels', 'length', 'answer_tokens'],
        num_rows: 8659
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels', 'length', 'answer_tokens'],
        num_rows: 1034
    })
})


In [ ]:

def check_split(split):
    lengths = np.array(tokenized_dataset[split]["length"])
    answer_tokens = np.array(tokenized_dataset[split]["answer_tokens"])
    return {
        "split": split,
        "count": len(lengths),
        "mean_length": lengths.mean(),
        "p95_length": np.percentile(lengths, 95),
        "max_length": lengths.max(),
        "mean_answer_tokens": answer_tokens.mean(),
        "zero_answer_tokens": int((answer_tokens == 0).sum()),
    }
pd.DataFrame([
    check_split("train"),
    check_split("validation"),
])

,split,count,mean_length,p95_length,max_length,mean_answer_tokens,zero_answer_tokens
0,train,8659,430.613928,904.0,2048,33.603649,82
1,validation,1034,330.686654,696.7,779,28.911992,0


In [11]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=None,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt",
)

In [ ]:
from trl import SFTTrainer, SFTConfig
sft_config = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,

    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_steps=5,

    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,

    fp16=True,
    bf16=False,
    

    optim="paged_adamw_8bit",
    weight_decay=0.01,
    max_grad_norm=0.3,

    report_to="none",
    dataloader_num_workers=4,           
    dataloader_pin_memory=True,          
    seed=322,
    data_seed=322,
)

In [14]:
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

print("Trainer is ready")

Trainer is ready


In [15]:
train_result = trainer.train()

print("Training finished")
print(train_result)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 8,659 | Num Epochs = 3 | Total steps = 1,626
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
50,0.203990,0.264709
100,0.174420,0.256117
150,0.144258,0.247732
200,0.136396,0.258777
250,0.125261,0.237847
300,0.117972,0.239888
350,0.118253,0.246875
400,0.102427,0.244896
450,0.125697,0.242506
500,0.086462,0.255780


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

KeyboardInterrupt: 

In [ ]:

def clean_generated_sql(full_output: str) -> str:
    if "### SQL:" in full_output:
        sql = full_output.split("### SQL:")[-1]
    else:
        sql = full_output

    sql = sql.strip()
    sql = sql.replace("```sql", "").replace("```", "").strip()
    for marker in ["###", "\nQuestion:", "\nDatabase schema:"]:
        if marker in sql:
            sql = sql.split(marker)[0].strip()

    return sql.strip()

def generate_sql(prompt: str, max_new_tokens: int = 256) -> str:
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return clean_generated_sql(decoded)
if DEV_EVAL_PATH.exists():
    with open(DEV_EVAL_PATH, "r", encoding="utf-8") as f:
        eval_example = json.loads(next(f))
    pred_sql = generate_sql(eval_example["text"])
    print("QUESTION:")
    print(eval_example["question"])
    print("\nPRED SQL:")
    print(pred_sql)
    print("\nGOLD SQL:")
    print(eval_example["sql"])
else:
    print("dev_eval_prompts.jsonl не найден, пропускаем генерацию.")

In [ ]:
N_DEV_PREDICTIONS = 50
PRED_DIR = OUTPUT_DIR / "predictions"
PRED_DIR.mkdir(parents=True, exist_ok=True)
PRED_PATH = PRED_DIR / f"dev_predictions_first_{N_DEV_PREDICTIONS}.jsonl"

if DEV_EVAL_PATH.exists():
    predictions = []
    with open(DEV_EVAL_PATH, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= N_DEV_PREDICTIONS:
                break

            item = json.loads(line)
            pred_sql = generate_sql(item["text"])

            predictions.append({
                "id": item.get("id"),
                "db_id": item.get("db_id"),
                "question": item.get("question"),
                "gold_sql": item.get("sql"),
                "pred_sql": pred_sql,
            })

    with open(PRED_PATH, "w", encoding="utf-8") as f:
        for item in predictions:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    print("Saved predictions to:", PRED_PATH)
    display(pd.DataFrame(predictions).head())

In [ ]:
from pathlib import Path

MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

ADAPTER_DIR = Path("/kaggle/input/models/olllllllll/lora16-1/pytorch/lora/1")
EVAL_PROMPTS_FILE = Path("/kaggle/input/datasets/olllllllll/spider/dev_eval_prompts.jsonl")



Adapter exists: True
Eval file exists: True
[PosixPath('/kaggle/input/models/olllllllll/lora16-1/pytorch/lora/1/adapter_model.safetensors'), PosixPath('/kaggle/input/models/olllllllll/lora16-1/pytorch/lora/1/adapter_config.json'), PosixPath('/kaggle/input/models/olllllllll/lora16-1/pytorch/lora/1/tokenizer.json'), PosixPath('/kaggle/input/models/olllllllll/lora16-1/pytorch/lora/1/tokenizer_config.json')]


In [ ]:

MAX_SEQ_LENGTH = 2048

base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
)

FastLanguageModel.for_inference(model)

print("Model with adapter loaded")

==((====))==  Unsloth 2026.6.1: Fast Qwen2 patching. Transformers: 5.10.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model with adapter loaded


In [ ]:
examples = []
with open(EVAL_PROMPTS_FILE, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 10:
            break
        examples.append(json.loads(line))
print("Loaded examples:", len(examples))
print(examples[0].keys())

Loaded examples: 10
dict_keys(['id', 'split', 'db_id', 'question', 'schema', 'sql', 'text'])


In [21]:
def clean_generated_sql(text: str) -> str:
    text = text.strip()

    if "```sql" in text:
        text = text.split("```sql", 1)[-1]
        text = text.split("```", 1)[0]
    elif "```" in text:
        text = text.split("```", 1)[-1]
        text = text.split("```", 1)[0]

    stop_markers = [
        "### Explanation:",
        "Explanation:",
        "\n\nExplanation",
        "\n###",
        "\nNote:",
        "\nThe query",
        "\nThis query",
    ]

    for marker in stop_markers:
        if marker in text:
            text = text.split(marker, 1)[0]

    text = text.strip()

    if ";" in text:
        text = text.split(";", 1)[0].strip()

    return text

In [ ]:
def generate_sql(example, max_new_tokens=128):
    prompt = example["text"]

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    raw_sql = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    clean_sql = clean_generated_sql(raw_sql)
    return raw_sql, clean_sql

In [ ]:
for i, ex in enumerate(examples):
    raw_pred, clean_pred = generate_sql(ex)
    print("=" * 120)
    print("EXAMPLE:", i)
    print("DB:", ex["db_id"])
    print("QUESTION:", ex["question"])

    print("\nRAW PRED:")
    print(raw_pred)

    print("\nCLEAN PRED:")
    print(clean_pred)

    print("\nGOLD:")
    print(ex["sql"])

Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


EXAMPLE: 0
DB: concert_singer
QUESTION: How many singers do we have?

RAW PRED:
SELECT count(*) FROM singer

CLEAN PRED:
SELECT count(*) FROM singer

GOLD:
SELECT count(*) FROM singer


Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


EXAMPLE: 1
DB: concert_singer
QUESTION: What is the total number of singers?

RAW PRED:
SELECT count(*) FROM singer

CLEAN PRED:
SELECT count(*) FROM singer

GOLD:
SELECT count(*) FROM singer


Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

EXAMPLE: 6
DB: concert_singer
QUESTION: Show the name and the release year of the song by the youngest singer.

RAW PRED:
SELECT T1.song_name , T1.song_release_year FROM singer AS T1 JOIN concert AS T2 ON T1.singer_id = T2.singer_id ORDER BY T1.age LIMIT 1

CLEAN PRED:
SELECT T1.song_name , T1.song_release_year FROM singer AS T1 JOIN concert AS T2 ON T1.singer_id = T2.singer_id ORDER BY T1.age LIMIT 1

GOLD:
SELECT song_name , song_release_year FROM singer ORDER BY age LIMIT 1


Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


EXAMPLE: 7
DB: concert_singer
QUESTION: What are the names and release years for all the songs of the youngest singer?

RAW PRED:
SELECT song_name , song_release_year FROM singer ORDER BY age LIMIT 1

CLEAN PRED:
SELECT song_name , song_release_year FROM singer ORDER BY age LIMIT 1

GOLD:
SELECT song_name , song_release_year FROM singer ORDER BY age LIMIT 1


Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


EXAMPLE: 8
DB: concert_singer
QUESTION: What are all distinct countries where singers above age 20 are from?

RAW PRED:
SELECT DISTINCT country FROM singer WHERE age > 20

CLEAN PRED:
SELECT DISTINCT country FROM singer WHERE age > 20

GOLD:
SELECT DISTINCT country FROM singer WHERE age > 20
EXAMPLE: 9
DB: concert_singer
QUESTION: What are  the different countries with singers above age 20?

RAW PRED:
SELECT DISTINCT country FROM singer WHERE age > 20

CLEAN PRED:
SELECT DISTINCT country FROM singer WHERE age > 20

GOLD:
SELECT DISTINCT country FROM singer WHERE age > 20


In [23]:
def normalize_sql_for_string_match(sql: str) -> str:
    sql = sql.strip().lower()
    sql = sql.replace(";", "")
    sql = " ".join(sql.split())
    return sql

N = 50
eval_examples = []

with open(EVAL_PROMPTS_FILE, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= N:
            break
        eval_examples.append(json.loads(line))

results = []

for ex in eval_examples:
    raw_pred, clean_pred = generate_sql(ex)

    pred_norm = normalize_sql_for_string_match(clean_pred)
    gold_norm = normalize_sql_for_string_match(ex["sql"])

    results.append({
        "id": ex["id"],
        "db_id": ex["db_id"],
        "question": ex["question"],
        "gold_sql": ex["sql"],
        "raw_pred_sql": raw_pred,
        "pred_sql": clean_pred,
        "string_exact_match": pred_norm == gold_norm,
    })

string_em = sum(r["string_exact_match"] for r in results) / len(results)

print("N:", len(results))
print("Simple string exact match:", round(string_em, 4))

Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

N: 50
Simple string exact match: 0.46


In [24]:
!git clone https://github.com/taoyds/spider.git /kaggle/working/spider_official

Cloning into '/kaggle/working/spider_official'...
remote: Enumerating objects: 386, done.
remote: Counting objects: 100% (73/73), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 386 (delta 45), reused 35 (delta 35), pack-reused 313 (from 1)
Receiving objects: 100% (386/386), 44.92 MiB | 35.14 MiB/s, done.
Resolving deltas: 100% (115/115), done.


In [ ]:
EVAL_PROMPTS_FILE = Path("/kaggle/input/datasets/olllllllll/spider/dev_eval_prompts.jsonl")

PREDICTIONS_DIR = Path("/kaggle/working/spider_predictions")
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

PREDICTIONS_JSONL = PREDICTIONS_DIR / "dev_predictions.jsonl"

In [ ]:

FastLanguageModel.for_inference(model)
model.eval()
examples = []

with open(EVAL_PROMPTS_FILE, "r", encoding="utf-8") as f:
    for line in f:
        examples.append(json.loads(line))

print("Dev examples:", len(examples))

with open(PREDICTIONS_JSONL, "w", encoding="utf-8") as f:
    for ex in tqdm(examples):
        raw_pred, clean_pred = generate_sql(ex)
        item = {
            "id": ex["id"],
            "db_id": ex["db_id"],
            "question": ex["question"],
            "gold_sql": ex["sql"],
            "raw_pred_sql": raw_pred,
            "pred_sql": clean_pred,
        }

        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Saved:", PREDICTIONS_JSONL)

Dev examples: 1034


  0%|          | 0/1034 [00:00<?, ?it/s]

Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

Saved: /kaggle/working/spider_predictions/dev_predictions.jsonl


In [34]:
EVAL_DIR = Path("/kaggle/working/spider_eval_files")
EVAL_DIR.mkdir(parents=True, exist_ok=True)

GOLD_SQL_PATH = EVAL_DIR / "gold.sql"
PRED_SQL_PATH = EVAL_DIR / "pred.sql"

def clean_for_eval(sql: str) -> str:
    sql = sql.strip()
    sql = sql.replace("\n", " ")
    sql = " ".join(sql.split())
    if sql.endswith(";"):
        sql = sql[:-1].strip()
    return sql

with open(PREDICTIONS_JSONL, "r", encoding="utf-8") as source, \
     open(GOLD_SQL_PATH, "w", encoding="utf-8") as gold_file, \
     open(PRED_SQL_PATH, "w", encoding="utf-8") as pred_file:

    for line in source:
        item = json.loads(line)

        gold_sql = clean_for_eval(item["gold_sql"])
        pred_sql = clean_for_eval(item["pred_sql"])
        db_id = item["db_id"]

        gold_file.write(f"{gold_sql}\t{db_id}\n")
        pred_file.write(f"{pred_sql}\n")

print("Gold:", GOLD_SQL_PATH)
print("Pred:", PRED_SQL_PATH)

Gold: /kaggle/working/spider_eval_files/gold.sql
Pred: /kaggle/working/spider_eval_files/pred.sql


In [36]:
SPIDER_DATA_DIR = Path("/kaggle/input/datasets/olllllllll/spider-orig/spider")
SPIDER_DB_DIR = SPIDER_DATA_DIR / "database"
SPIDER_TABLES = SPIDER_DATA_DIR / "tables.json"

print("DB dir exists:", SPIDER_DB_DIR.exists(), SPIDER_DB_DIR)
print("Tables exists:", SPIDER_TABLES.exists(), SPIDER_TABLES)

DB dir exists: True /kaggle/input/datasets/olllllllll/spider-orig/spider/database
Tables exists: True /kaggle/input/datasets/olllllllll/spider-orig/spider/tables.json


In [38]:
!python /kaggle/working/spider_official/evaluation.py \
  --gold /kaggle/working/spider_eval_files/gold.sql \
  --pred /kaggle/working/spider_eval_files/pred.sql \
  --db {SPIDER_DB_DIR} \
  --table {SPIDER_TABLES} \
  --etype all

eval_err_num:1
medium pred: SELECT T1.song_name , T1.song_release_year FROM singer AS T1 JOIN concert AS T2 ON T1.singer_id = T2.singer_id ORDER BY T1.age LIMIT 1
medium gold: SELECT song_name , song_release_year FROM singer ORDER BY age LIMIT 1

medium pred: SELECT max(capacity) , avg(capacity) FROM stadium
medium gold: select max(capacity), average from stadium

medium pred: SELECT count(*) , T1.name FROM stadium AS T1 JOIN concert AS T2 ON T1.stadium_id = T2.stadium_id GROUP BY T1.name
medium gold: SELECT T2.name , count(*) FROM concert AS T1 JOIN stadium AS T2 ON T1.stadium_id = T2.stadium_id GROUP BY T1.stadium_id

extra pred: SELECT T1.name , T1.capacity FROM stadium AS T1 JOIN concert AS T2 ON T1.stadium_id = T2.stadium_id WHERE YEAR >= 2014 GROUP BY T1.name ORDER BY count(*) DESC LIMIT 1
extra gold: SELECT T2.name , T2.capacity FROM concert AS T1 JOIN stadium AS T2 ON T1.stadium_id = T2.stadium_id WHERE T1.year >= 2014 GROUP BY T2.stadium_id ORDER BY count(*) DESC LIMIT 1

hard

In [39]:
!git clone https://github.com/taoyds/test-suite-sql-eval.git /kaggle/working/test_suite_eval

Cloning into '/kaggle/working/test_suite_eval'...
remote: Enumerating objects: 61, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 61 (delta 20), reused 16 (delta 16), pack-reused 31 (from 2)
Receiving objects: 100% (61/61), 619.62 KiB | 10.68 MiB/s, done.
Resolving deltas: 100% (24/24), done.


In [40]:
!python /kaggle/working/test_suite_eval/evaluation.py \
  --gold /kaggle/working/spider_eval_files/gold.sql \
  --pred /kaggle/working/spider_eval_files/pred.sql \
  --db {SPIDER_DB_DIR} \
  --table {SPIDER_TABLES} \
  --etype all

/kaggle/working/test_suite_eval/exec_eval.py:127: SyntaxWarning: invalid escape sequence '\s'
  "YEAR\s*\(\s*CURDATE\s*\(\s*\)\s*\)\s*", "2020", query, flags=re.IGNORECASE
/kaggle/working/test_suite_eval/parse.py:57: SyntaxWarning: invalid escape sequence '\d'
  float_nums = re.findall("[-+]?\d*\.\d+", query)
/kaggle/working/test_suite_eval/parse.py:62: SyntaxWarning: invalid escape sequence '\d'
  int_nums = [i.strip() for i in re.findall("[^tT]\d+", query)]
/kaggle/working/test_suite_eval/parse.py:70: SyntaxWarning: invalid escape sequence '\d'
  table = re.findall("[Tt]\d+\.", tok)
/kaggle/working/test_suite_eval/parse.py:206: SyntaxWarning: invalid escape sequence '\.'
  for table, col, val1, val2 in re.findall('(?:([^\.\s]*)\.)?([^\.\s]+) between ([^\s;]+) and ([^\s;]+)', query, re.IGNORECASE):
medium pred: SELECT T1.song_name , T1.song_release_year FROM singer AS T1 JOIN concert AS T2 ON T1.singer_id = T2.singer_id ORDER BY T1.age LIMIT 1
medium gold: SELECT song_name , song_rele